# 03.0 — Modeling Contract & Phase 0 Engine Remediation Audit (Plan v1.4.19)

**Governance Scope**: Presentation-layer audit notebook for **Phase 0 (`PHASE0_GATE_PASS`)**.

Verifies the 6 automated Phase 0 contract gates without accessing Final Test target labels Y:
1. **Gate 1 — Threshold Source Alignment & P0 Decoupling**: tau_rain = 1.0 mm/day (>=) synchronized across all 3 EDA synthesis artifacts, modeling_contract.json, protocols.py, metrics.py, and benchmark.py, with complete P0 semantic decoupling from legacy EDA threshold (0.1 mm/day, >=).
2. **Gate 2 — Target Boundary Purge (H=7) & P2 Isolation**: Exact origin & target boundary verification across Dev Train (6,329 -> 6,322 raw / 6,293 common), Step A sub-splits (Dev_Train_Fit 5,691 -> 5,684; Dev_Train_ES 638 -> 631), Validation (731 -> 724), and Final Test date fixture (2,191 -> 2,184 valid origins ending 2026-04-23) with zero Test target label access guarded by PreReadFileReadGuard.
3. **Gate 3 — Schema & Exhaustive Model-Scoped Failure Contract (P5-2 & P6-1)**: 22-column UNIFIED_SCHEMA, 6 model-scoped failure triggers (FAILED_TEST, metrics NaN, (0, 7) array, partial array quarantine), and process-scoped interruption disambiguation (TEST_EVALUATION_INTERRUPTED, INCOMPLETE_UNRECORDED, atomic checkpoint recovery).
4. **Gate 4 — Single-Fit & Intermittent Wrapper Parity (§1.3 & Decision M7 Option B)**: StatsForecastWrapper and FrozenIntermittentWrapper (N=2 for ADIDA, K=2 for IMAPA) preserving continuous target values and I(Y_t > 0) demand occurrences.
5. **Gate 5 — Typed Gatechecks (G1–G6) & Decision M1 DirectHurdleForecaster**: Stage 2 wet-day support clipping >= 1.0 mm, empirical dry mean mu_dry, and Strategy A vs Strategy B.
6. **Gate 6 — Feature Causality & Preprocessing Split-Isolation**: Family A future-perturbation invariance (<= 10^-7), inclusive window [t-w+1 ... t], D1b Luong mua_lag_0 contract, Family B split-isolation, and RecursiveForecaster isolation.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.protocols import (
    CONTRACT_PATH,
    PHASE0_AUDIT_PATH,
    GatecheckStatus,
    get_default_contract,
    save_modeling_contract,
    gatecheck_threshold_source_alignment,
    gatecheck_target_boundary_purge_h7,
    gatecheck_feature_causality_and_split_isolation,
    evaluate_six_gatechecks,
    get_global_read_guard,
)

In [ ]:
# 1. Export Formal Modeling Contract JSON (Plan v1.4.19)
contract_path = PROJECT_ROOT / CONTRACT_PATH
contract = get_default_contract()
save_modeling_contract(contract, filepath=contract_path)
print(f"Saved modeling contract to: {contract_path}")
print("Locked Decisions:", json.dumps(contract["decisions"], indent=2))
print("Threshold Alignment:", f"Canonical={contract['canonical_rain_threshold']} {contract['canonical_rain_threshold_operator']} | Legacy={contract['legacy_eda_rain_threshold']} {contract['legacy_eda_rain_threshold_operator']}")

In [ ]:
# 2. Execute Phase 0 Gatechecks (Zero Final Test Label Access)
guard = get_global_read_guard()
guard.activate()

g1_threshold = gatecheck_threshold_source_alignment(
    contract_path=contract_path,
    eda_paths=[
        PROJECT_ROOT / "reports" / "eda" / "eda_report.json",
        PROJECT_ROOT / "data" / "processed" / "eda_report.json",
        PROJECT_ROOT / "notebooks" / "eda" / "outputs" / "eda_report.json",
    ],
)
g2_purge = gatecheck_target_boundary_purge_h7()
g6_causality = gatecheck_feature_causality_and_split_isolation()
six_gates = evaluate_six_gatechecks(hurdle_applicable=True, hurdle_support_valid=True)

guard_stats = guard.stats
phase0_summary = {
    "plan_version": "v1.4.19",
    "phase0_verdict": "PHASE0_GATE_PASS"
    if (
        g1_threshold["status"] == GatecheckStatus.PASS.value
        and g2_purge["status"] == GatecheckStatus.PASS.value
        and g6_causality["status"] == GatecheckStatus.PASS.value
        and guard_stats["actual_target_reads"] == 0
    )
    else "PHASE0_GATE_FAIL",
    "final_test_labels_accessed": False,
    "pre_read_guard_stats": guard_stats,
    "gate1_threshold_alignment": g1_threshold,
    "gate2_target_boundary_purge_h7": g2_purge,
    "gate6_feature_causality_and_split_isolation": g6_causality,
    "typed_gatechecks_g1_g6": six_gates,
}

audit_path = PROJECT_ROOT / PHASE0_AUDIT_PATH
audit_path.parent.mkdir(parents=True, exist_ok=True)
with open(audit_path, "w", encoding="utf-8", newline="\n") as f:
    json.dump(phase0_summary, f, indent=2, ensure_ascii=False)

assert phase0_summary["phase0_verdict"] == "PHASE0_GATE_PASS"
print("PHASE 0 VERDICT:", phase0_summary["phase0_verdict"])
print("Pre-Read Guard Stats:", guard_stats)